In [ ]:
import pandas as pd
import numpy as np 
import os

In [ ]:
EXP_NUMBER = 1
NUM_DATA_POINTS = 10000

**load the data**

In [ ]:

#read the json file
data_file_path = "../data/processed/multinli_1.0_train_cleaned.csv"
data = pd.read_csv(data_file_path, sep='µ')

#get a random sample of the data
data = data.sample(n = NUM_DATA_POINTS, random_state = 42)
data.head()

**clean the texts fields**

### Apply TF-IDF

In [ ]:
import spacy
import re
import unicodedata
from tqdm import tqdm

def clean_text(text):
    # Remove accented characters (optional)
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    
    # Remove anything that is NOT a letter or space
    text = re.sub(r"[^a-zA-Z\s]", ' ', text)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Lowercase the text
    text = text.lower()
    
    return text

nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"]  # disable unused components
)


def preprocess_texts(texts, batch_size=10000):
    cleaned = []

    # First, clean numbers/special chars
    texts = [clean_text(t) for t in texts]

    for doc in tqdm(nlp.pipe(texts, batch_size=batch_size), total=len(texts)):
        tokens = [
            token.lemma_.lower()
            for token in doc
        ]
        cleaned.append(" ".join(tokens))

    return cleaned

In [ ]:
preprocess_texts(['not no  does, never, neither, nobody, nothing ,nowhere, none, nor, cannot, t, without, hardly, scarcely, barely, seldom, rarely trying to test the code. All every each always completely entirely totally definitely certainly surely'])

In [ ]:
#clean text sentence1 and sentence2
data['sentence1_cleaned'] = preprocess_texts(data['sentence1'].astype(str).tolist())
data['sentence2_cleaned'] = preprocess_texts(data['sentence2'].astype(str).tolist())

In [ ]:
#apply tf-idf, remove stop words, remove numbering
from sklearn.feature_extraction.text import TfidfVectorizer

all_sentences = data['sentence1_cleaned'].tolist() + data['sentence2_cleaned'].tolist()

# Fit on all text to get the same vocabulary
tfidf = TfidfVectorizer()
tfidf.fit(all_sentences)

# Transform each column separately using the same vectorizer
tfidf_s1_matrix = tfidf.transform(data['sentence1_cleaned']).toarray()
tfidf_s2_matrix = tfidf.transform(data['sentence2_cleaned']).toarray()

In [ ]:
#use the tfidf vectors as features (concatenate sentence1 and sentence2)
feature_names_s1 = [f"s1_{w}" for w in tfidf.get_feature_names_out()]
feature_names_s2 = [f"s2_{w}" for w in tfidf.get_feature_names_out()]
all_feature_names = feature_names_s1 + feature_names_s2

features = pd.DataFrame(
    np.hstack([tfidf_s1_matrix, tfidf_s2_matrix]),
    columns=all_feature_names
)

#add the features to the cleaned_data
data_tf_idf = pd.concat([data.reset_index(drop=True), features.reset_index(drop=True)], axis=1)

#drop sentence1 and sentence2
data_tf_idf = data_tf_idf.drop(columns=['sentence1', 'sentence2', 'genre', 'sentence1_cleaned', 'sentence2_cleaned', 'annotator_labels'])
data_tf_idf.head()

In [ ]:
type(data_tf_idf)

### feature engineering

### train models

In [ ]:
#split the data into train and test
from sklearn.model_selection import train_test_split

#drop columns annotator_labels gold_label
cleaned_data_train = data_tf_idf.drop(columns=['gold_label'])

X_train, X_test, y_train, y_test = train_test_split(cleaned_data_train, data_tf_idf['gold_label'], test_size=0.2, random_state=42)

**dimesion reduction for distance base algorithmes**

In [ ]:
from sklearn.decomposition import TruncatedSVD

# Fit SVD with enough components
svd = TruncatedSVD(n_components=min(X_train.shape[1], 5000), random_state=42)
X_train_svd = svd.fit_transform(X_train)
X_test_svd = svd.transform(X_test)

# Compute cumulative explained variance
cumulative_variance = np.cumsum(svd.explained_variance_ratio_)

# Choose number of components to explain at least 70% variance
n_components_70 = np.searchsorted(cumulative_variance, 0.7) + 1
print(f"Number of components to retain 70% variance: {n_components_70}")

# Refit SVD with optimal components
svd_opt = TruncatedSVD(n_components=n_components_70, random_state=42)
X_train_svd = svd_opt.fit_transform(X_train)
X_test_svd = svd_opt.transform(X_test)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import json

models = {
    "Random Forest": RandomForestClassifier(n_jobs=-1),
    "Logistic Regression": LogisticRegression(multi_class='multinomial', n_jobs=-1),
    "SVM": SVC(kernel='linear')
}

for model_name, model in models.items():
    print(f"Training {model_name}...")

    if model_name in {"SVM", "Logistic Regression"}:
        model.fit(X_train_svd, y_train)
        pred = model.predict(X_test_svd)
    else:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

    # --- get class names ---
    class_names = model.classes_

    # --- per-class metrics as dict ---
    per_class_metrics = {}
    for cls in class_names:
        per_class_metrics[cls] = {
            "precision": float(precision_score(y_test, pred, average=None, labels=[cls])[0]),
            "recall": float(recall_score(y_test, pred, average=None, labels=[cls])[0]),
            "f1": float(f1_score(y_test, pred, average=None, labels=[cls])[0])
        }

    # --- overall metrics ---
    accuracy = accuracy_score(y_test, pred)
    precision_weighted = precision_score(y_test, pred, average='weighted')
    recall_weighted = recall_score(y_test, pred, average='weighted')
    f1_weighted = f1_score(y_test, pred, average='weighted')

    # --- save model ---
    os.makedirs(f"../artifacts/{EXP_NUMBER}/model", exist_ok=True)
    joblib.dump(model, f"../artifacts/{EXP_NUMBER}/model/{model_name.replace(' ', '_')}_tfidf_model.pkl")
    
    os.makedirs(f"../artifacts/{EXP_NUMBER}/vectorizer", exist_ok=True)
    joblib.dump(tfidf, f"../artifacts/{EXP_NUMBER}/vectorizer/tfidf_vectorizer.pkl")

    # save metrics json
    metrics = {
        "per_class_metrics": per_class_metrics,
        "accuracy": accuracy,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted
    }
    
    os.makedirs(f"../artifacts/{EXP_NUMBER}/metrics", exist_ok=True)
    
    #confusion matrix
    cm = confusion_matrix(y_test, pred, labels=class_names)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(f'Confusion Matrix for {model_name}')
    plt.savefig(f"../artifacts/{EXP_NUMBER}/metrics/{model_name.replace(' ', '_')}_confusion_matrix.png")
    plt.close()
    

    with open(f"../artifacts/{EXP_NUMBER}/metrics/{model_name.replace(' ', '_')}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=4)
